In [3]:
# Notebook 04 (v2): Train/Val/Test split with live progress visibility.
# Inputs : CSV.zip on Drive
# Outputs: train.parquet, val.parquet, test.parquet, sample_500k.parquet
# Runtime: ~15-25 min on Colab Pro+ A100 + High-RAM

import os, sys, time, shutil, zipfile, gc, json, subprocess
from pathlib import Path
from multiprocessing import Pool, cpu_count
import numpy as np
import pandas as pd

# Install tqdm if missing (Colab has it but be defensive)
try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'])
    from tqdm.auto import tqdm

from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

PROJECT     = Path('/content/drive/MyDrive/composite_resilience_framework')
RAW_DIR     = PROJECT / 'data' / 'raw'
PROCESSED   = PROJECT / 'data' / 'processed'
RESULTS     = PROJECT / 'results'
PROCESSED.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

LOCAL_SCRATCH = Path('/content/scratch')
LOCAL_SCRATCH.mkdir(exist_ok=True)
DATA_ROOT     = LOCAL_SCRATCH / 'CICIoT2023_CSV' / 'CSV'

SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

def mem_gb():
    """Current process RSS in GB."""
    out = subprocess.run(['free', '-g'], capture_output=True, text=True).stdout
    line = [l for l in out.split('\n') if l.startswith('Mem:')][0].split()
    used = int(line[2])
    avail = int(line[6]) if len(line) >= 7 else int(line[3])
    return used, avail

def log(msg):
    used, avail = mem_gb()
    ts = time.strftime('%H:%M:%S')
    print(f"[{ts}] {msg}   (RAM used {used}G, avail {avail}G)", flush=True)

# =========================================================================
# Pre-flight: verify we're on the right runtime
# =========================================================================
print("=" * 70)
print("PRE-FLIGHT: Runtime check")
print("=" * 70)
total_ram_gb = int(subprocess.run(['free', '-g'], capture_output=True, text=True)
                   .stdout.split('\n')[1].split()[1])
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(f"  Total RAM: {total_ram_gb} GB")
print(f"  GPU: {gpu if gpu else 'NONE'}")
if total_ram_gb < 50:
    print(f"\n  ⚠ WARNING: only {total_ram_gb} GB RAM. Expected ~83 GB on A100+HighRAM.")
    print(f"  The dataset load will likely OOM. Stop now and fix the runtime first:")
    print(f"    Runtime → Disconnect and delete runtime")
    print(f"    Runtime → Change runtime type → A100 GPU → Save")
    print(f"    Re-run this cell")
    raise SystemExit("Halting on insufficient RAM.")
print(f"  ✓ Runtime looks correct — proceeding\n")

# =========================================================================
# STEP 1: ensure dataset is extracted locally
# =========================================================================
print("=" * 70)
print("STEP 1: Ensure dataset is extracted locally")
print("=" * 70)
if not DATA_ROOT.exists():
    log("Copying CSV.zip from Drive to local scratch...")
    t0 = time.time()
    shutil.copy(RAW_DIR / 'CSV.zip', LOCAL_SCRATCH / 'CSV.zip')
    log(f"Copied in {time.time()-t0:.1f}s. Extracting...")
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_SCRATCH / 'CSV.zip') as z:
        members = z.namelist()
        for m in tqdm(members, desc="Extracting", unit="file"):
            z.extract(m, LOCAL_SCRATCH / 'CICIoT2023_CSV')
    log(f"Extracted in {time.time()-t0:.1f}s")
else:
    log("Dataset already extracted locally — skipping")

# =========================================================================
# STEP 2: parallel-load all 309 CSVs with live progress
# =========================================================================
print("\n" + "=" * 70)
print("STEP 2: Load all CSVs in parallel with live progress")
print("=" * 70)

def categorize(n):
    if n.startswith('DDoS-'): return 'DDoS'
    if n.startswith('DoS-'): return 'DoS'
    if n.startswith('Mirai-'): return 'Mirai'
    if n.startswith('Recon-') or n == 'VulnerabilityScan': return 'Recon'
    if n == 'Benign_Final': return 'Benign'
    if n == 'DictionaryBruteForce': return 'BruteForce'
    if n in ('DNS_Spoofing', 'MITM-ArpSpoofing'): return 'Spoofing'
    if n in ('Backdoor_Malware','BrowserHijacking','CommandInjection',
             'SqlInjection','Uploading_Attack','XSS'): return 'Web'
    raise ValueError(f"Uncategorized folder: {n}")

work = []
for cls_folder in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    coarse = categorize(cls_folder.name)
    for csv_path in sorted(cls_folder.glob('*.csv')):
        work.append((str(csv_path), cls_folder.name, coarse))
log(f"Total CSVs to load: {len(work)}")

def load_one(args):
    path, fine, coarse = args
    df = pd.read_csv(path)
    df['_fine_class']   = fine
    df['_category']     = coarse
    return df

t0 = time.time()
n_workers = min(cpu_count(), 8)
log(f"Starting parallel load with {n_workers} workers...")
parts = []
with Pool(processes=n_workers) as pool:
    # imap_unordered yields results as workers finish — enables live progress
    for df_part in tqdm(pool.imap_unordered(load_one, work),
                        total=len(work), desc="Loading CSVs", unit="file"):
        parts.append(df_part)
log(f"All {len(parts)} files loaded in {time.time()-t0:.1f}s")

log("Concatenating into one DataFrame (this takes ~30-90s)...")
t0 = time.time()
df = pd.concat(parts, ignore_index=True, copy=False)
del parts
gc.collect()
log(f"Concatenated in {time.time()-t0:.1f}s — shape {df.shape}")

log(f"Memory of combined DataFrame: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

log("Downcasting numeric dtypes to reduce memory...")
t0 = time.time()
float_cols = df.select_dtypes(include=['float64']).columns
int_cols   = df.select_dtypes(include=['int64']).columns
for col in tqdm(float_cols, desc="Downcast floats"):
    df[col] = pd.to_numeric(df[col], downcast='float')
for col in tqdm(int_cols, desc="Downcast ints"):
    df[col] = pd.to_numeric(df[col], downcast='integer')
df['_fine_class'] = df['_fine_class'].astype('category')
df['_category']   = df['_category'].astype('category')
gc.collect()
log(f"Downcast in {time.time()-t0:.1f}s")
log(f"Memory after downcast: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# =========================================================================
# STEP 3: verify class distribution
# =========================================================================
print("\n" + "=" * 70)
print("STEP 3: Verify class distribution")
print("=" * 70)
cat_counts = df['_category'].value_counts().sort_values(ascending=False)
total = len(df)
print(f"\n  Total rows: {total:,}\n")
print(f"  {'Category':<12} {'Rows':>15} {'% of total':>12}")
print(f"  {'-'*12} {'-'*15} {'-'*12}")
for cat, n in cat_counts.items():
    print(f"  {cat:<12} {n:>15,} {100*n/total:>11.2f}%")

expected_total = 46_776_700
if total == expected_total:
    print(f"\n  ✓ Total rows match expected ({expected_total:,})")
else:
    print(f"\n  ⚠ Total = {total:,}, expected {expected_total:,}, delta {total - expected_total:+,}")

# =========================================================================
# STEP 4: stratified 70/15/15 split
# =========================================================================
print("\n" + "=" * 70)
print("STEP 4: Stratified 70/15/15 split on 8-category label")
print("=" * 70)
from sklearn.model_selection import train_test_split

log("Splitting train (70%) vs temp (30%)...")
t0 = time.time()
idx_train, idx_temp = train_test_split(
    np.arange(len(df)),
    test_size=0.30, stratify=df['_category'].values, random_state=SEED,
)
log(f"  done in {time.time()-t0:.1f}s")

log("Splitting temp into val (15%) and test (15%)...")
t0 = time.time()
idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50, stratify=df['_category'].values[idx_temp], random_state=SEED,
)
log(f"  done in {time.time()-t0:.1f}s")

print(f"\n  Split sizes:")
print(f"    Train: {len(idx_train):>12,} ({100*len(idx_train)/total:5.2f}%)")
print(f"    Val:   {len(idx_val):>12,} ({100*len(idx_val)/total:5.2f}%)")
print(f"    Test:  {len(idx_test):>12,} ({100*len(idx_test)/total:5.2f}%)")

# =========================================================================
# STEP 5: verify stratification
# =========================================================================
print("\n" + "=" * 70)
print("STEP 5: Verify stratification per split")
print("=" * 70)
splits = {'Train': idx_train, 'Val': idx_val, 'Test': idx_test}
checks = []
for name, idx in splits.items():
    vc = df.iloc[idx]['_category'].value_counts(normalize=True).sort_index()
    checks.append(vc.rename(name))
prop_table = pd.concat(checks, axis=1) * 100
prop_table = prop_table.reindex(cat_counts.index)
print("\n  Per-category % in each split (should match across columns):")
print(prop_table.to_string(float_format='%.3f'))

max_drift = (prop_table.max(axis=1) - prop_table.min(axis=1)).max()
print(f"\n  Max drift across splits: {max_drift:.4f} percentage points")
assert max_drift < 0.05, f"Stratification drift too large: {max_drift}"
print("  ✓ Stratification verified")

# =========================================================================
# STEP 6: write Parquet splits to Drive (with heartbeat)
# =========================================================================
print("\n" + "=" * 70)
print("STEP 6: Write Parquet splits to Drive")
print("=" * 70)
print("  (each write takes 2-5 min — Drive write is the bottleneck)\n")

def save_split(name, idx):
    out = PROCESSED / f'{name}.parquet'
    log(f"  Writing {name} ({len(idx):,} rows) → {out.name}...")
    t0 = time.time()
    df.iloc[idx].to_parquet(out, engine='pyarrow', compression='snappy', index=False)
    sz = out.stat().st_size / 1024**3
    log(f"    {name} done: {sz:.2f} GB on Drive in {time.time()-t0:.1f}s")

save_split('train', idx_train)
save_split('val',   idx_val)
save_split('test',  idx_test)

# =========================================================================
# STEP 7: build 500k stratified mini-sample
# =========================================================================
print("\n" + "=" * 70)
print("STEP 7: Build 500k stratified mini-sample")
print("=" * 70)
SAMPLE_N = 500_000
MIN_PER_CAT = 200

cat_counts_total = df['_category'].value_counts()
sample_idx_parts = []
rng = np.random.default_rng(SEED)
for cat in cat_counts_total.index:
    cat_idx = np.where(df['_category'].values == cat)[0]
    target = max(MIN_PER_CAT, int(SAMPLE_N * cat_counts_total[cat] / total))
    target = min(target, len(cat_idx))
    chosen = rng.choice(cat_idx, size=target, replace=False)
    sample_idx_parts.append(chosen)
sample_idx = np.concatenate(sample_idx_parts)
rng.shuffle(sample_idx)
sample_idx = sample_idx[:SAMPLE_N]

log(f"Writing sample_500k.parquet...")
sample_df = df.iloc[sample_idx].copy()
out_sample = PROCESSED / 'sample_500k.parquet'
sample_df.to_parquet(out_sample, engine='pyarrow', compression='snappy', index=False)
log(f"  Sample saved: {out_sample.stat().st_size/1024**2:.1f} MB")
print(f"\n  Sample category distribution:")
print(sample_df['_category'].value_counts(normalize=True).sort_index().map('{:.4f}'.format).to_string())

# =========================================================================
# STEP 8: write manifest
# =========================================================================
manifest = {
    'seed': SEED,
    'total_rows': int(total),
    'train_rows': int(len(idx_train)),
    'val_rows':   int(len(idx_val)),
    'test_rows':  int(len(idx_test)),
    'sample_rows': int(len(sample_df)),
    'categories': cat_counts.index.tolist(),
    'feature_columns': [c for c in df.columns if not c.startswith('_')],
    'label_column_coarse': '_category',
    'label_column_fine': '_fine_class',
}
with open(RESULTS / 'split_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
log(f"Wrote {RESULTS / 'split_manifest.json'}")

print("\n" + "=" * 70)
print("DONE — splits saved to Drive at:")
print(f"  {PROCESSED}")
print("=" * 70)

PRE-FLIGHT: Runtime check
  Total RAM: 167 GB
  GPU: NVIDIA A100-SXM4-80GB, 81920 MiB
  ✓ Runtime looks correct — proceeding

STEP 1: Ensure dataset is extracted locally
[20:38:22] Copying CSV.zip from Drive to local scratch...   (RAM used 1G, avail 164G)
[20:38:23] Copied in 1.6s. Extracting...   (RAM used 1G, avail 164G)


Extracting:   0%|          | 0/345 [00:00<?, ?file/s]

[20:39:06] Extracted in 42.7s   (RAM used 1G, avail 164G)

STEP 2: Load all CSVs in parallel with live progress
[20:39:06] Total CSVs to load: 309   (RAM used 1G, avail 164G)
[20:39:06] Starting parallel load with 8 workers...   (RAM used 1G, avail 164G)


Loading CSVs:   0%|          | 0/309 [00:00<?, ?file/s]

[20:39:53] All 309 files loaded in 47.4s   (RAM used 15G, avail 149G)
[20:39:53] Concatenating into one DataFrame (this takes ~30-90s)...   (RAM used 15G, avail 149G)
[20:40:13] Concatenated in 19.7s — shape (46776700, 41)   (RAM used 17G, avail 148G)
[20:40:32] Memory of combined DataFrame: 18.71 GB   (RAM used 17G, avail 148G)
[20:40:32] Downcasting numeric dtypes to reduce memory...   (RAM used 17G, avail 148G)


Downcast floats:   0%|          | 0/38 [00:00<?, ?it/s]

Downcast ints:   0%|          | 0/1 [00:00<?, ?it/s]

[20:41:14] Downcast in 42.1s   (RAM used 10G, avail 155G)
[20:41:14] Memory after downcast: 7.28 GB   (RAM used 10G, avail 155G)

STEP 3: Verify class distribution

  Total rows: 46,776,700

  Category                Rows   % of total
  ------------ --------------- ------------
  DDoS              33,984,450       72.65%
  DoS                7,845,120       16.77%
  Mirai              2,634,054        5.63%
  Benign             1,098,191        2.35%
  Recon                690,534        1.48%
  Spoofing             486,458        1.04%
  Web                   24,829        0.05%
  BruteForce            13,064        0.03%

  ✓ Total rows match expected (46,776,700)

STEP 4: Stratified 70/15/15 split on 8-category label
[20:41:15] Splitting train (70%) vs temp (30%)...   (RAM used 10G, avail 155G)
[20:42:06]   done in 50.5s   (RAM used 10G, avail 154G)
[20:42:06] Splitting temp into val (15%) and test (15%)...   (RAM used 10G, avail 154G)
[20:42:22]   done in 15.9s   (RAM used 10G, ava